### style_fabric_preference_model_comparison

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import os
import json
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# Import scikit-learn modules
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, SplineTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.base import BaseEstimator, TransformerMixin
import statsmodels.api as sm
from scipy import stats
from xgboost import XGBRegressor
import lightgbm as lgb
from pygam import LinearGAM


# ==================== File reading ====================
excel_path = 'style_fabric_preference.xlsx'

print(f"Data file path: {excel_path}")
print(f"File exists: {os.path.exists(excel_path)}")

if not os.path.exists(excel_path):
    raise FileNotFoundError(
        f"File not found: style_fabric_preference.xlsx\n"
        f"Current directory: {os.getcwd()}\n"
        f"Please check the file location."
    )

# Read style data set
df_style = pd.read_excel(excel_path, sheet_name='style_data')
df_style.columns = ['id', 'style', 'preference_mean', 'search_index', 'info_index']

# Read fabric data set
df_fabric = pd.read_excel(excel_path, sheet_name='fabric_data')
df_fabric.columns = ['id', 'fabric', 'preference_mean', 'search_index', 'info_index']

print("Style data shape:", df_style.shape)
print("Fabric data shape:", df_fabric.shape)


# ==================== 2. Data preprocessing ====================
def preprocess_data(df):
    """Clean and transform the data."""
    # Convert to numeric, coercing errors
    df['search_index'] = pd.to_numeric(df['search_index'], errors='coerce')
    df['info_index'] = pd.to_numeric(df['info_index'], errors='coerce')
    df['preference_mean'] = pd.to_numeric(df['preference_mean'], errors='coerce')

    # Drop rows with missing values in key columns
    df = df.dropna(subset=['search_index', 'info_index', 'preference_mean']).copy()

    # Log-transform independent variables (add small epsilon to avoid log(0))
    df['ln_search'] = np.log(df['search_index'] + 1e-10)
    df['ln_info'] = np.log(df['info_index'] + 1e-10)

    # Dependent variable keeps its raw value
    return df


df_style = preprocess_data(df_style)
df_fabric = preprocess_data(df_fabric)

print(f"Style data after cleaning: {df_style.shape[0]} samples")
print(f"Fabric data after cleaning: {df_fabric.shape[0]} samples")

# ==================== 3. Prepare data for modeling ====================
# Style data
X_style_single_search = df_style[['ln_search']].values
X_style_single_info = df_style[['ln_info']].values
X_style_double = df_style[['ln_search', 'ln_info']].values
y_style = df_style['preference_mean'].values

# Fabric data
X_fabric_single_search = df_fabric[['ln_search']].values
X_fabric_single_info = df_fabric[['ln_info']].values
X_fabric_double = df_fabric[['ln_search', 'ln_info']].values
y_fabric = df_fabric['preference_mean'].values

print(f"\nStyle data: X_single_search shape={X_style_single_search.shape}, "
      f"X_double shape={X_style_double.shape}, y shape={y_style.shape}")
print(f"Fabric data: X_single_search shape={X_fabric_single_search.shape}, "
      f"X_double shape={X_fabric_double.shape}, y shape={y_fabric.shape}")


# ==================== 4. Linear regression significance analysis ====================
def analyze_linear_significance(X, y, feature_names=None, model_name=""):
    """
    Fit an OLS linear regression and analyze coefficient significance.
    Returns a dictionary with coefficients, p-values, t-values, etc.
    """
    results = {}

    try:
        # Add a constant term for the intercept
        X_with_const = sm.add_constant(X)

        # Fit the OLS model
        model = sm.OLS(y, X_with_const)
        fitted_model = model.fit()

        # Extract coefficients and statistics
        coefficients = fitted_model.params
        p_values = fitted_model.pvalues
        t_values = fitted_model.tvalues
        std_err = fitted_model.bse
        conf_int_array = np.asarray(fitted_model.conf_int())

        summary = {}

        # Retrieve variable names from the fitted model
        if hasattr(fitted_model, 'model') and hasattr(fitted_model.model, 'exog_names'):
            var_names = fitted_model.model.exog_names
        else:
            var_names = ['const'] + (feature_names if feature_names else
                                     [f'X{i}' for i in range(1, len(coefficients))])

        # Ensure name/length consistency
        n_params = len(coefficients)
        if len(var_names) != n_params:
            if len(var_names) > n_params:
                var_names = var_names[:n_params]
            else:
                var_names = var_names + [f'X{i}' for i in range(len(var_names), n_params)]

        for i in range(n_params):
            var_name = var_names[i]
            if var_name == 'const':
                var_name = 'Intercept'

            summary[var_name] = {
                "coefficient": float(coefficients[i]),
                "std_error": float(std_err[i]),
                "t_value": float(t_values[i]),
                "p_value": float(p_values[i]),
                "significant": bool(p_values[i] < 0.05),
                "conf_lower": float(conf_int_array[i, 0]),
                "conf_upper": float(conf_int_array[i, 1])
            }

        # Overall model statistics
        results["summary"] = summary
        results["r_squared"] = float(fitted_model.rsquared)
        results["adj_r_squared"] = float(fitted_model.rsquared_adj)
        results["f_statistic"] = float(fitted_model.fvalue)
        results["f_p_value"] = float(fitted_model.f_pvalue)
        results["aic"] = float(fitted_model.aic)
        results["bic"] = float(fitted_model.bic)

        # Classify significant variables
        significant_vars = [var for var, stats_ in summary.items()
                            if var != "Intercept" and stats_["p_value"] < 0.05]
        results["significant_variables"] = significant_vars
        results["n_significant"] = len(significant_vars)

    except Exception as e:
        results["error"] = str(e)

    return results


# ==================== 5. Model evaluation with regularization ====================
def evaluate_model_with_regularization(model, X, y, model_name, feature_type,
                                       param_grid=None, cv_folds=5):
    """
    Evaluate a model with optional hyperparameter tuning.
    Returns (results_dict, best_model).
    """
    results = {}

    # Hyperparameter tuning (if a grid is provided)
    if param_grid is not None and len(param_grid) > 0:
        # Use randomized search when the grid is large
        if np.prod([len(v) for v in param_grid.values()]) > 20:
            search = RandomizedSearchCV(
                model, param_grid, cv=cv_folds, scoring='r2',
                n_iter=10, random_state=42, n_jobs=-1, verbose=0
            )
            search_method = 'randomized'
        else:
            search = GridSearchCV(
                model, param_grid, cv=cv_folds, scoring='r2',
                n_jobs=-1, verbose=0
            )
            search_method = 'grid'

        print(f"    Performing {search_method} search...")
        search.fit(X, y)
        best_model = search.best_estimator_
        best_params = search.best_params_
        best_score = search.best_score_
        results['best_params'] = str(best_params)
        results['best_cv_score'] = best_score
        print(f"    Best params: {best_params}")
        print(f"    Best CV R2: {best_score:.4f}")
    else:
        best_model = model
        results['best_params'] = 'default'
        results['best_cv_score'] = np.nan

    # Cross-validation on the best model
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)

    cv_r2 = cross_val_score(best_model, X, y, cv=kf, scoring='r2')
    cv_rmse = -cross_val_score(best_model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    cv_mae = -cross_val_score(best_model, X, y, cv=kf, scoring='neg_mean_absolute_error')

    # Fit on the full training set
    best_model.fit(X, y)
    y_pred = best_model.predict(X)

    # Training metrics
    train_r2 = r2_score(y, y_pred)
    train_rmse = np.sqrt(mean_squared_error(y, y_pred))
    train_mae = mean_absolute_error(y, y_pred)

    # Store results
    results['model'] = model_name
    results['features'] = feature_type
    results['cv_r2_mean'] = cv_r2.mean()
    results['cv_r2_std'] = cv_r2.std()
    results['cv_rmse_mean'] = cv_rmse.mean()
    results['cv_rmse_std'] = cv_rmse.std()
    results['cv_mae_mean'] = cv_mae.mean()
    results['cv_mae_std'] = cv_mae.std()
    results['train_r2'] = train_r2
    results['train_rmse'] = train_rmse
    results['train_mae'] = train_mae
    results['n_samples'] = X.shape[0]
    results['n_features'] = X.shape[1]

    # Feature importance for tree-based models
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        results['feature_importance'] = str(
            dict(zip(['ln_search', 'ln_info'], importances.tolist()))
        )
    else:
        results['feature_importance'] = 'N/A'

    # Overfitting detection
    train_cv_gap = train_r2 - cv_r2.mean()
    results['train_cv_gap'] = train_cv_gap

    if train_cv_gap > 0.2:
        results['overfitting_status'] = 'Severe overfitting'
        overfitting_warning = f"[Severe overfitting] train-CV gap={train_cv_gap:.4f}"
    elif train_cv_gap > 0.1:
        results['overfitting_status'] = 'Moderate overfitting'
        overfitting_warning = f"[Moderate overfitting] train-CV gap={train_cv_gap:.4f}"
    elif train_cv_gap > 0.05:
        results['overfitting_status'] = 'Mild overfitting'
        overfitting_warning = f"[Mild overfitting] train-CV gap={train_cv_gap:.4f}"
    else:
        results['overfitting_status'] = 'Normal'
        overfitting_warning = f"[Normal] train-CV gap={train_cv_gap:.4f}"

    results['overfitting_warning'] = overfitting_warning

    return results, best_model


# ==================== 6. Custom transformers for spline regression ====================
class DualSplineTransformer(BaseEstimator, TransformerMixin):
    """Apply cubic spline transforms to two variables separately."""

    def __init__(self, degree=3, n_knots=4):
        self.degree = degree
        self.n_knots = n_knots
        self.spline_search = SplineTransformer(degree=degree, n_knots=n_knots,
                                               include_bias=False)
        self.spline_info = SplineTransformer(degree=degree, n_knots=n_knots,
                                             include_bias=False)

    def fit(self, X, y=None):
        # X shape: (n_samples, 2)
        self.spline_search.fit(X[:, [0]])
        self.spline_info.fit(X[:, [1]])
        return self

    def transform(self, X):
        X_search_transformed = self.spline_search.transform(X[:, [0]])
        X_info_transformed = self.spline_info.transform(X[:, [1]])
        return np.hstack([X_search_transformed, X_info_transformed])


class DualSplineWithInteraction(BaseEstimator, TransformerMixin):
    """Cubic spline transforms on two variables plus interaction terms."""

    def __init__(self, degree=3, n_knots=3):
        self.degree = degree
        self.n_knots = n_knots
        self.spline_search = SplineTransformer(degree=degree, n_knots=n_knots,
                                               include_bias=False)
        self.spline_info = SplineTransformer(degree=degree, n_knots=n_knots,
                                             include_bias=False)

    def fit(self, X, y=None):
        self.spline_search.fit(X[:, [0]])
        self.spline_info.fit(X[:, [1]])
        return self

    def transform(self, X):
        X_search_transformed = self.spline_search.transform(X[:, [0]])
        X_info_transformed = self.spline_info.transform(X[:, [1]])

        # Interaction terms: search-spline basis x info-spline basis
        interaction = np.zeros(
            (X.shape[0], X_search_transformed.shape[1] * X_info_transformed.shape[1])
        )
        idx = 0
        for i in range(X_search_transformed.shape[1]):
            for j in range(X_info_transformed.shape[1]):
                interaction[:, idx] = X_search_transformed[:, i] * X_info_transformed[:, j]
                idx += 1
        return np.hstack([X_search_transformed, X_info_transformed, interaction])


# ==================== 7. Model configurations ====================
def get_model_configs_enhanced(X_shape):
    """Define all models with enhanced regularization."""
    n_samples, n_features = X_shape

    # Enhanced regularization for tree models
    rf_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 4],
        'min_samples_split': [10, 20],
        'min_samples_leaf': [5, 10],
        'max_features': ['sqrt', 0.5],
        'bootstrap': [True]
    }

    lgb_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 4],
        'num_leaves': [7, 15],
        'learning_rate': [0.01, 0.05],
        'subsample': [0.7, 0.8],
        'colsample_bytree': [0.7, 0.8],
        'reg_alpha': [0.1, 0.5, 1.0],
        'reg_lambda': [0.1, 0.5, 1.0],
        'min_child_samples': [20, 50]
    }

    xgb_param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 4],
        'learning_rate': [0.01, 0.05],
        'subsample': [0.7, 0.8],
        'colsample_bytree': [0.7, 0.8],
        'reg_alpha': [0.1, 0.5, 1.0],
        'reg_lambda': [0.1, 0.5, 1.0]
    }

    # Regularized linear models
    ridge_param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
    lasso_param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0]}
    elastic_param_grid = {
        'alpha': [0.001, 0.01, 0.1, 1.0],
        'l1_ratio': [0.1, 0.5, 0.9]
    }

    # SVR parameter grid (kept for reference; inline grids are used below)
    svr_param_grid = {
        'C': [0.1, 1.0, 10.0, 100.0],
        'epsilon': [0.01, 0.1, 0.5],
        'gamma': ['scale', 'auto', 0.1, 1.0]
    }

    model_configs = []

    # ========== other models ==========
    # 1-3. Linear models
    model_configs.append(('Linear (search)', LinearRegression(), 'single_search', {}))
    model_configs.append(('Linear (info)', LinearRegression(), 'single_info', {}))
    model_configs.append(('Linear (both)', LinearRegression(), 'double', {}))

    # 4-6. Regularized linear models
    model_configs.append(('Ridge (both)', Ridge(random_state=42), 'double', ridge_param_grid))
    model_configs.append(('Lasso (both)', Lasso(random_state=42), 'double', lasso_param_grid))
    model_configs.append(('ElasticNet (both)', ElasticNet(random_state=42), 'double', elastic_param_grid))

    # 7-9. Polynomial + Ridge (original)
    poly2_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly2_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly2+Ridge (search)', poly2_ridge_pipeline,
                          'single_search_poly2', poly2_ridge_param_grid))

    poly3_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly3_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly3+Ridge (search)', poly3_ridge_pipeline,
                          'single_search_poly3', poly3_ridge_param_grid))

    poly2_both_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly2_both_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly2+Ridge (both)', poly2_both_ridge_pipeline,
                          'double_poly2', poly2_both_ridge_param_grid))

    # 10. Polynomial degree 2 on info
    poly2_info_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly2_info_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly2+Ridge (info)', poly2_info_ridge_pipeline,
                          'single_info_poly2', poly2_info_ridge_param_grid))

    # 11. Polynomial degree 3 on info
    poly3_info_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly3_info_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly3+Ridge (info)', poly3_info_ridge_pipeline,
                          'single_info_poly3', poly3_info_ridge_param_grid))

    # 12. Polynomial degree 3 on both variables
    poly3_both_ridge_pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False, interaction_only=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    poly3_both_ridge_param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Poly3+Ridge (both)', poly3_both_ridge_pipeline,
                          'double_poly3', poly3_both_ridge_param_grid))

    # 13. SVR with RBF kernel
    svr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf'))
    ])
    svr_rbf_param_grid = {
        'svr__C': [0.1, 1.0, 10.0, 100.0],
        'svr__epsilon': [0.01, 0.1, 0.5],
        'svr__gamma': ['scale', 'auto', 0.1, 1.0]
    }
    model_configs.append(('SVR (both)', svr_pipeline, 'double', svr_rbf_param_grid))

    # 14. SVR with linear kernel
    svr_linear_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='linear'))
    ])
    svr_linear_param_grid = {
        'svr__C': [0.1, 1.0, 10.0, 100.0],
        'svr__epsilon': [0.01, 0.1, 0.5]
    }
    model_configs.append(('SVR-Linear (both)', svr_linear_pipeline, 'double', svr_linear_param_grid))

    # 15. SVR with polynomial kernel (degree 2)
    svr_poly_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='poly', degree=2))
    ])
    svr_poly_param_grid = {
        'svr__C': [0.1, 1.0, 10.0, 100.0],
        'svr__epsilon': [0.01, 0.1, 0.5],
        'svr__gamma': ['scale', 'auto', 0.1],
        'svr__coef0': [0.0, 1.0]
    }
    model_configs.append(('SVR-Poly2 (both)', svr_poly_pipeline, 'double', svr_poly_param_grid))

    # 16. Random Forest
    model_configs.append(('Random Forest (reg)',
                          RandomForestRegressor(random_state=42, bootstrap=True),
                          'double', rf_param_grid))

    # 17. LightGBM
    model_configs.append(('LightGBM (reg)',
                          lgb.LGBMRegressor(random_state=42, verbose=-1),
                          'double', lgb_param_grid))

    # 18. XGBoost
    model_configs.append(('XGBoost (reg)',
                          XGBRegressor(random_state=42),
                          'double', xgb_param_grid))

    # 19. Cubic spline regression (single variable: search)
    spline_search_pipeline = Pipeline([
        ('spline', SplineTransformer(degree=3, n_knots=4, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    spline_search_param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Spline3+Ridge (search)', spline_search_pipeline,
                          'single_search', spline_search_param_grid))

    # 20. Cubic spline regression (single variable: info)
    spline_info_pipeline = Pipeline([
        ('spline', SplineTransformer(degree=3, n_knots=4, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    spline_info_param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Spline3+Ridge (info)', spline_info_pipeline,
                          'single_info', spline_info_param_grid))

    # 21. Cubic spline regression (both variables)
    dual_spline_pipeline = Pipeline([
        ('dual_spline', DualSplineTransformer(degree=3, n_knots=4)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    dual_spline_param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Spline3+Ridge (both)', dual_spline_pipeline,
                          'double', dual_spline_param_grid))

    # 22. Cubic spline regression (both variables, with interactions)
    dual_spline_interact_pipeline = Pipeline([
        ('dual_spline', DualSplineWithInteraction(degree=3, n_knots=3)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(random_state=42))
    ])
    dual_spline_interact_param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
    model_configs.append(('Spline3+Ridge+Interact (both)', dual_spline_interact_pipeline,
                          'double', dual_spline_interact_param_grid))

    # 23. GAM
    model_configs.append(('GAM', LinearGAM(), 'double', {}))

    return model_configs


# ==================== 8. Significance analysis helper ====================
def perform_significance_analysis(X_dict, y, dataset_name):
    """Run significance analysis for linear regression models."""
    print(f"\n{'=' * 40}")
    print(f"{dataset_name.upper()} DATA - SIGNIFICANCE ANALYSIS")
    print(f"{'=' * 40}")

    significance_results = {}

    # Single variable: ln_search
    print(f"\n1. Single variable: ln_search")
    X_single = X_dict['single_search']
    feature_names = ['ln_search']
    sig_result = analyze_linear_significance(X_single, y, feature_names, "Linear (search)")

    if 'error' not in sig_result:
        print(f"   R-squared: {sig_result['r_squared']:.4f}")
        print(f"   Adjusted R-squared: {sig_result['adj_r_squared']:.4f}")
        print(f"   F-statistic: {sig_result['f_statistic']:.2f} (p={sig_result['f_p_value']:.4f})")

        for var, stats_ in sig_result['summary'].items():
            significance = ("***" if stats_['p_value'] < 0.001
                            else "**" if stats_['p_value'] < 0.01
                            else "*" if stats_['p_value'] < 0.05 else "")
            print(f"   {var}: coef={stats_['coefficient']:.6f}, "
                  f"std err={stats_['std_error']:.6f}, t={stats_['t_value']:.3f}, "
                  f"p={stats_['p_value']:.6f} {significance}")

        significance_results['single_search'] = sig_result
    else:
        print(f"   Error: {sig_result['error']}")

    # Single variable: ln_info
    print(f"\n2. Single variable: ln_info")
    X_single = X_dict['single_info']
    feature_names = ['ln_info']
    sig_result = analyze_linear_significance(X_single, y, feature_names, "Linear (info)")

    if 'error' not in sig_result:
        print(f"   R-squared: {sig_result['r_squared']:.4f}")
        print(f"   Adjusted R-squared: {sig_result['adj_r_squared']:.4f}")
        print(f"   F-statistic: {sig_result['f_statistic']:.2f} (p={sig_result['f_p_value']:.4f})")

        for var, stats_ in sig_result['summary'].items():
            significance = ("***" if stats_['p_value'] < 0.001
                            else "**" if stats_['p_value'] < 0.01
                            else "*" if stats_['p_value'] < 0.05 else "")
            print(f"   {var}: coef={stats_['coefficient']:.6f}, "
                  f"std err={stats_['std_error']:.6f}, t={stats_['t_value']:.3f}, "
                  f"p={stats_['p_value']:.6f} {significance}")

        significance_results['single_info'] = sig_result
    else:
        print(f"   Error: {sig_result['error']}")

    # Both variables
    print(f"\n3. Both variables: ln_search + ln_info")
    X_double = X_dict['double']
    feature_names = ['ln_search', 'ln_info']
    sig_result = analyze_linear_significance(X_double, y, feature_names, "Linear (both)")

    if 'error' not in sig_result:
        print(f"   R-squared: {sig_result['r_squared']:.4f}")
        print(f"   Adjusted R-squared: {sig_result['adj_r_squared']:.4f}")
        print(f"   F-statistic: {sig_result['f_statistic']:.2f} (p={sig_result['f_p_value']:.4f})")
        print(f"   Significant variables ({sig_result['n_significant']}/2): "
              f"{sig_result['significant_variables']}")

        for var, stats_ in sig_result['summary'].items():
            significance = ("***" if stats_['p_value'] < 0.001
                            else "**" if stats_['p_value'] < 0.01
                            else "*" if stats_['p_value'] < 0.05 else "")
            print(f"   {var}: coef={stats_['coefficient']:.6f}, "
                  f"std err={stats_['std_error']:.6f}, t={stats_['t_value']:.3f}, "
                  f"p={stats_['p_value']:.6f} {significance}")

        significance_results['double'] = sig_result
    else:
        print(f"   Error: {sig_result['error']}")

    return significance_results


# ==================== 9. Run model comparison ====================
print("\n" + "=" * 80)
print("MODEL TRAINING AND EVALUATION")
print("=" * 80)


def run_model_comparison(X_dict, y, dataset_name):
    """Run the complete model comparison for one dataset."""
    print(f"\n{'=' * 40}")
    print(f"{dataset_name.upper()} DATA")
    print(f"{'=' * 40}")

    all_results = []
    trained_models = {}

    model_configs = get_model_configs_enhanced(X_dict['double'].shape)

    for model_name, model, feature_type, param_grid in model_configs:
        print(f"\nTraining {model_name}...")

        # Select the appropriate feature set
        if feature_type == 'single_search':
            X = X_dict['single_search']
        elif feature_type == 'single_info':
            X = X_dict['single_info']
        elif feature_type == 'double':
            X = X_dict['double']
        elif feature_type == 'single_search_poly2':
            X = X_dict['single_search']
        elif feature_type == 'single_search_poly3':
            X = X_dict['single_search']
        elif feature_type == 'single_info_poly2':
            X = X_dict['single_info']
        elif feature_type == 'single_info_poly3':
            X = X_dict['single_info']
        elif feature_type == 'double_poly2':
            X = X_dict['double']
        elif feature_type == 'double_poly3':
            X = X_dict['double']
        else:
            X = X_dict['double']

        # Special handling for GAM
        if model_name == 'GAM':
            try:
                gam = LinearGAM()
                gam.fit(X, y)
                y_pred = gam.predict(X)

                train_r2 = r2_score(y, y_pred)
                train_rmse = np.sqrt(mean_squared_error(y, y_pred))
                train_mae = mean_absolute_error(y, y_pred)

                kf = KFold(n_splits=5, shuffle=True, random_state=42)
                cv_r2_scores = []
                cv_rmse_scores = []
                cv_mae_scores = []

                for train_idx, val_idx in kf.split(X):
                    X_train, X_val = X[train_idx], X[val_idx]
                    y_train, y_val = y[train_idx], y[val_idx]
                    gam_cv = LinearGAM()
                    gam_cv.fit(X_train, y_train)
                    y_val_pred = gam_cv.predict(X_val)

                    cv_r2_scores.append(r2_score(y_val, y_val_pred))
                    cv_rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_val_pred)))
                    cv_mae_scores.append(mean_absolute_error(y_val, y_val_pred))

                cv_r2_mean = np.mean(cv_r2_scores)
                cv_r2_std = np.std(cv_r2_scores)
                cv_rmse_mean = np.mean(cv_rmse_scores)
                cv_rmse_std = np.std(cv_rmse_scores)
                cv_mae_mean = np.mean(cv_mae_scores)
                cv_mae_std = np.std(cv_mae_scores)
                train_cv_gap = train_r2 - cv_r2_mean

                if train_cv_gap > 0.2:
                    overfitting_status = 'Severe overfitting'
                    overfitting_warning = f"[Severe overfitting] train-CV gap={train_cv_gap:.4f}"
                elif train_cv_gap > 0.1:
                    overfitting_status = 'Moderate overfitting'
                    overfitting_warning = f"[Moderate overfitting] train-CV gap={train_cv_gap:.4f}"
                elif train_cv_gap > 0.05:
                    overfitting_status = 'Mild overfitting'
                    overfitting_warning = f"[Mild overfitting] train-CV gap={train_cv_gap:.4f}"
                else:
                    overfitting_status = 'Normal'
                    overfitting_warning = f"[Normal] train-CV gap={train_cv_gap:.4f}"

                results = {
                    'model': model_name,
                    'features': 'double',
                    'cv_r2_mean': cv_r2_mean,
                    'cv_r2_std': cv_r2_std,
                    'cv_rmse_mean': cv_rmse_mean,
                    'cv_rmse_std': cv_rmse_std,
                    'cv_mae_mean': cv_mae_mean,
                    'cv_mae_std': cv_mae_std,
                    'train_r2': train_r2,
                    'train_rmse': train_rmse,
                    'train_mae': train_mae,
                    'n_samples': X.shape[0],
                    'n_features': X.shape[1],
                    'best_params': 'default',
                    'best_cv_score': np.nan,
                    'feature_importance': 'N/A',
                    'train_cv_gap': train_cv_gap,
                    'overfitting_status': overfitting_status,
                    'overfitting_warning': overfitting_warning
                }
                all_results.append(results)
                trained_models[model_name] = gam
                print(f"  Train R2: {train_r2:.4f}, CV R2: {cv_r2_mean:.4f} ± {cv_r2_std:.4f} "
                      f"- {overfitting_status}")
                continue
            except Exception as e:
                print(f"  GAM training failed: {e}")
                continue

        # Standard models
        try:
            results, best_model = evaluate_model_with_regularization(
                model, X, y, model_name, feature_type, param_grid
            )
            all_results.append(results)
            trained_models[model_name] = best_model
            print(f"  Train R2: {results['train_r2']:.4f}, "
                  f"CV R2: {results['cv_r2_mean']:.4f} ± {results['cv_r2_std']:.4f} "
                  f"- {results['overfitting_warning']}")
        except Exception as e:
            print(f"  Training failed: {e}")

    results_df = pd.DataFrame(all_results)
    if len(results_df) > 0:
        results_df = results_df.sort_values('cv_r2_mean', ascending=False).reset_index(drop=True)
    else:
        print(f"Warning: no model was successfully trained for {dataset_name}")

    return results_df, trained_models


# Prepare feature dictionaries
X_style_dict = {
    'single_search': X_style_single_search,
    'single_info': X_style_single_info,
    'double': X_style_double
}

X_fabric_dict = {
    'single_search': X_fabric_single_search,
    'single_info': X_fabric_single_info,
    'double': X_fabric_double
}

# Significance analysis first
print("\n" + "=" * 80)
print("LINEAR REGRESSION SIGNIFICANCE ANALYSIS")
print("=" * 80)

significance_style = perform_significance_analysis(X_style_dict, y_style, 'Style')
significance_fabric = perform_significance_analysis(X_fabric_dict, y_fabric, 'Fabric')

# Model comparisons
results_style, models_style = run_model_comparison(X_style_dict, y_style, 'Style')
results_fabric, models_fabric = run_model_comparison(X_fabric_dict, y_fabric, 'Fabric')

# ==================== 10. Results summary ====================
print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

if len(results_style) > 0:
    results_style['dataset'] = 'style'
if len(results_fabric) > 0:
    results_fabric['dataset'] = 'fabric'

key_columns = ['dataset', 'model', 'features', 'cv_r2_mean', 'cv_r2_std',
               'train_r2', 'train_cv_gap', 'overfitting_status', 'train_rmse',
               'cv_rmse_mean', 'n_samples', 'best_params']

combined_results = pd.concat(
    [results_style[key_columns] if len(results_style) > 0 else pd.DataFrame(),
     results_fabric[key_columns] if len(results_fabric) > 0 else pd.DataFrame()],
    ignore_index=True
)

if len(combined_results) > 0:
    combined_results = combined_results.sort_values(['dataset', 'cv_r2_mean'],
                                                    ascending=[True, False])
    print("\nAll models sorted by dataset and CV R2:")
    print(combined_results.to_string())
else:
    print("No results available")

print("\n" + "-" * 80)
print("BEST MODEL PER DATASET")
print("-" * 80)

for name, res_df, sig_dict in [('STYLE', results_style, significance_style),
                                ('FABRIC', results_fabric, significance_fabric)]:
    print(f"\n=== {name} ===")
    if len(res_df) == 0:
        print("No results available")
        continue

    best = res_df.iloc[0]
    print(f"Best model: {best['model']} ({best['features']})")
    print(f"  CV R2: {best['cv_r2_mean']:.4f} ± {best['cv_r2_std']:.4f}")
    print(f"  Train R2: {best['train_r2']:.4f}, Train RMSE: {best['train_rmse']:.4f}")
    print(f"  train-CV gap: {best['train_cv_gap']:.4f} - {best['overfitting_status']}")
    if best['best_params'] != 'default':
        print(f"  Best params: {best['best_params']}")

    if 'double' in sig_dict and 'error' not in sig_dict['double']:
        sig = sig_dict['double']
        print(f"  Significance: R2={sig['r_squared']:.4f}, "
              f"Adj R2={sig['adj_r_squared']:.4f}, "
              f"F={sig['f_statistic']:.2f} (p={sig['f_p_value']:.6f})")
        print(f"  Significant variables: {sig['significant_variables']} "
              f"({sig['n_significant']}/2)")


# ==================== 11. Save results ====================
print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

output_dir = 'model_results_v2'

try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory: {output_dir}")
except Exception as e:
    print(f"Failed to create directory: {e}")
    output_dir = os.getcwd()
    print(f"Using fallback directory: {output_dir}")

try:
    # CSV files
    style_path = os.path.join(output_dir, 'style_model_results.csv')
    fabric_path = os.path.join(output_dir, 'fabric_model_results.csv')
    combined_path = os.path.join(output_dir, 'combined_model_results.csv')

    if len(results_style) > 0:
        results_style.to_csv(style_path, index=False, encoding='utf-8-sig')
        print(f"  - {style_path}")
    if len(results_fabric) > 0:
        results_fabric.to_csv(fabric_path, index=False, encoding='utf-8-sig')
        print(f"  - {fabric_path}")
    if len(combined_results) > 0:
        combined_results.to_csv(combined_path, index=False, encoding='utf-8-sig')
        print(f"  - {combined_path}")

    print(f"CSV files saved")

    # JSON files for significance results
    significance_style_path = os.path.join(output_dir, 'style_significance.json')
    significance_fabric_path = os.path.join(output_dir, 'fabric_significance.json')

    def convert_to_serializable(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.bool_):
            return bool(obj)
        else:
            return obj

    def convert_dict(d):
        if isinstance(d, dict):
            return {k: convert_dict(v) for k, v in d.items()}
        elif isinstance(d, list):
            return [convert_dict(item) for item in d]
        else:
            return convert_to_serializable(d)

    with open(significance_style_path, 'w', encoding='utf-8') as f:
        json.dump(convert_dict(significance_style), f, ensure_ascii=False, indent=2)

    with open(significance_fabric_path, 'w', encoding='utf-8') as f:
        json.dump(convert_dict(significance_fabric), f, ensure_ascii=False, indent=2)

    print(f"JSON files saved")

except Exception as e:
    print(f"Error while saving files: {e}")
    print("Attempting to save to the current directory...")
    if len(results_style) > 0:
        results_style.to_csv('style_model_results.csv', index=False, encoding='utf-8-sig')
    if len(results_fabric) > 0:
        results_fabric.to_csv('fabric_model_results.csv', index=False, encoding='utf-8-sig')
    if len(combined_results) > 0:
        combined_results.to_csv('combined_model_results.csv', index=False, encoding='utf-8-sig')
    print("Files saved to the current directory")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETED SUCCESSFULLY!")
print("=" * 80)